# Session 7 — Developing and Deploying APIs for ML Models

**Goal:** take a trained classifier out of a notebook and put it behind a real HTTP
API — a FastAPI service with validated request/response schemas, input checks that
reject bad data before it ever reaches the model, and a versioning scheme so the API
and the model underneath it can evolve independently.

## What this session automates

Session 4 trained a model and got predictions by calling `endpoint.predict()` from
Python — convenient, but only usable by someone with GCP credentials and the
`aiplatform` SDK installed. A REST API removes that dependency: any client that can
send an HTTP request (a web frontend, a mobile app, a curl command, another
microservice) can get a prediction, with no knowledge of how the model was trained
or what's running behind it. This session builds that wrapper by hand with
**FastAPI** and **Pydantic**, so you can see exactly what a managed endpoint like
Vertex AI's is doing for you under the hood.

## The dataset

This session uses the UCI **Heart Disease** dataset (Cleveland subset, 303
patients, 13 clinical features — age, resting blood pressure, cholesterol, max
heart rate, ST depression, and so on — plus a binary target for presence of heart
disease). It's a good fit for an API walkthrough for a boring but important reason:
the features are individually meaningful numbers a client actually has to type into
a request (age, blood pressure, cholesterol), which makes input validation — the
main new idea in this session — concrete rather than abstract.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* names the
exact line or value to look at in that cell's output; *Infer* says what conclusion
to draw from it, and what a different result would imply. Cells that define the API
itself (the FastAPI app, the Pydantic models) are written to a `.py` file with
`%%writefile`, then exercised with `TestClient` and `curl` examples — the same code
you'd run with `uvicorn` in production.

## Prerequisites

```bash
pip install fastapi uvicorn pydantic scikit-learn pandas ucimlrepo httpx
```

No cloud account is needed for this session — everything runs locally. The `httpx`
package is what FastAPI's `TestClient` uses under the hood to send requests to the
app in-process, without actually opening a network socket.

## Step 1 — Fetch the dataset

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart = fetch_ucirepo(id=45)
X = heart.data.features
y = heart.data.targets

df = pd.concat([X, y], axis=1)
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

**Observe:** the printed shape (`303 rows, 14 columns`) and the preview — the
last column is `num`, the UCI target, coded `0` (no disease) through `4`
(increasing severity). Also notice `ca` and `thal` may show `NaN` for a handful of
rows — this dataset has a small number of missing values recorded as `?` in the
original source, which `ucimlrepo` already coerces to `NaN`.
**Infer:** the multi-level `num` target is more granularity than a first API
needs — the next cell collapses it to a binary "disease present / absent" label,
which is both the standard way this dataset is used in practice and a simpler
contract for API clients (`0`/`1` instead of five ambiguous severity levels). If
you saw only two columns here instead of fourteen, `fetch_ucirepo` returned targets
only, not features — check the `id=45` argument.

In [ ]:
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"]).dropna()
print(df["target"].value_counts())
print(f"Rows after dropping missing values: {len(df)}")

**Observe:** the class counts (roughly 160 negative / 137 positive on the real
UCI split) and the row count after `dropna()` (297 of the original 303 — six rows
dropped for missing `ca`/`thal` values).
**Infer:** a fairly balanced binary target means plain accuracy is a reasonable
headline metric later, without needing to reach for class weighting or a metric
like F1 to compensate for imbalance. Losing six rows to `dropna()` is small enough
to ignore here, but in a real pipeline you'd log that number — silently dropping
20% of a dataset because of missing values would be a signal to impute rather than
drop.

## Step 2 — Train the classifier the API will serve

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib

FEATURE_COLUMNS = [c for c in df.columns if c != "target"]
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURE_COLUMNS], df["target"], test_size=0.2, random_state=42, stratify=df["target"]
)

clf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
print(f"ROC AUC : {roc_auc_score(y_test, proba):.3f}")

joblib.dump(clf, "heart_disease_model.joblib")
print("Saved heart_disease_model.joblib")

**Observe:** the accuracy and ROC AUC lines — a real run on this split scores
roughly **accuracy 0.850, ROC AUC 0.912** — and the final `Saved` confirmation.
**Infer:** this is a plain scikit-learn model, trained and evaluated exactly like
Sessions 1-3 — the API work starting in Step 3 is entirely orthogonal to *how* the
model was produced. `joblib.dump` is what makes the model loadable by a separate
process (the API server) without re-running training — if this file is missing or
stale, the API will silently serve predictions from an old model version, which is
exactly the versioning problem Step 7 addresses.

## Step 3 — Define the request/response schema with Pydantic

A `BaseModel` subclass is both documentation and enforcement: FastAPI uses it to
generate the OpenAPI schema (the interactive docs at `/docs`) *and* to validate
every incoming request body before your handler function ever runs.

In [ ]:
%%writefile schemas.py
from pydantic import BaseModel, Field


class PatientFeatures(BaseModel):
    age: float = Field(..., ge=1, le=120, description="Age in years")
    sex: int = Field(..., ge=0, le=1, description="1 = male, 0 = female")
    cp: int = Field(..., ge=0, le=3, description="Chest pain type (0-3)")
    trestbps: float = Field(..., ge=60, le=250, description="Resting blood pressure (mm Hg)")
    chol: float = Field(..., ge=100, le=600, description="Serum cholesterol (mg/dl)")
    fbs: int = Field(..., ge=0, le=1, description="Fasting blood sugar > 120 mg/dl")
    restecg: int = Field(..., ge=0, le=2, description="Resting ECG results (0-2)")
    thalach: float = Field(..., ge=60, le=250, description="Max heart rate achieved")
    exang: int = Field(..., ge=0, le=1, description="Exercise-induced angina")
    oldpeak: float = Field(..., ge=0, le=10, description="ST depression induced by exercise")
    slope: int = Field(..., ge=0, le=2, description="Slope of the peak exercise ST segment")
    ca: float = Field(..., ge=0, le=4, description="Number of major vessels colored by fluoroscopy")
    thal: float = Field(..., ge=0, le=7, description="Thalassemia test result")


class PredictionResponse(BaseModel):
    prediction: int
    probability: float
    label: str
    model_version: str

**Observe:** the `Writing schemas.py` confirmation, and the `Field(..., ge=..., le=...)`
bounds on every feature — each one comes from the plausible physiological range for
that measurement, not an arbitrary guess (e.g. resting blood pressure between 60
and 250 mm Hg).
**Infer:** these bounds are the first line of defense against bad input — a client
that sends `age=-5` or `trestbps=9999` (a unit mistake, a null sent as a sentinel
value, a typo) gets rejected with a `422 Unprocessable Entity` *before* it ever
reaches `clf.predict()`, which would otherwise happily return a meaningless
prediction for nonsense input rather than erroring. `...` (Ellipsis) as the default
marks a field required — a request missing any one of these thirteen features is
also rejected automatically, with no `if` statements written by hand.

## Step 4 — Build the FastAPI app

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from schemas import PatientFeatures, PredictionResponse

MODEL_VERSION = "v1.0.0"
MODEL_PATH = "heart_disease_model.joblib"

app = FastAPI(
    title="Heart Disease Prediction API",
    version=MODEL_VERSION,
    description="Predicts presence of heart disease from clinical features (UCI Heart Disease dataset).",
)

model = joblib.load(MODEL_PATH)
FEATURE_ORDER = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal",
]


@app.get("/health")
def health():
    return {"status": "ok", "model_version": MODEL_VERSION}


@app.post("/v1/predict", response_model=PredictionResponse)
def predict(features: PatientFeatures):
    try:
        row = pd.DataFrame([features.model_dump()])[FEATURE_ORDER]
        proba = float(model.predict_proba(row)[0, 1])
        pred = int(proba >= 0.5)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Inference failed: {exc}")

    return PredictionResponse(
        prediction=pred,
        probability=round(proba, 4),
        label="disease" if pred == 1 else "no_disease",
        model_version=MODEL_VERSION,
    )

**Observe:** the `Writing app.py` confirmation, and specifically the route path
`/v1/predict` (not just `/predict`) and the module-level `model = joblib.load(...)`
call, which happens once at import time rather than inside the handler.
**Infer:** loading the model at import time (not per-request) is what makes this
API fast — deserializing a joblib file on every single request would add tens of
milliseconds of latency and disk I/O to every prediction for no benefit, since the
model doesn't change between requests. The `/v1/` prefix is the versioning
decision this session cares about: it reserves room for a `/v2/predict` later that
can change the request schema entirely without breaking clients still pointed at
`/v1/` — see Step 7.

## Step 5 — Exercise the API locally with `TestClient`

`TestClient` drives the FastAPI app in-process — no `uvicorn` server, no open port
— which makes it the right tool for fast local testing and for the automated tests
a CI pipeline (Session 10) would run on every push.

In [ ]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

resp = client.get("/health")
print(resp.status_code, resp.json())

**Observe:** `200 {'status': 'ok', 'model_version': 'v1.0.0'}`.
**Infer:** a successful `/health` call confirms two things at once — the app
imported without error (no syntax mistakes in `app.py` or `schemas.py`) and the
model file loaded successfully. This is the exact endpoint a deployment platform
(Kubernetes, SageMaker, Cloud Run) would poll to decide whether this container is
ready to receive traffic — if it doesn't build and run inside a wrapper container
per Session 6's pattern before you push it, `/health` failing is what would keep a
real deployment stuck at 0 ready replicas.

In [ ]:
sample_patient = {
    "age": 63, "sex": 1, "cp": 3, "trestbps": 145, "chol": 233,
    "fbs": 1, "restecg": 0, "thalach": 150, "exang": 0,
    "oldpeak": 2.3, "slope": 0, "ca": 0, "thal": 1,
}

resp = client.post("/v1/predict", json=sample_patient)
print(resp.status_code)
print(resp.json())

**Observe:** `200`, followed by a JSON body like
`{'prediction': 1, 'probability': 0.7833, 'label': 'disease', 'model_version': 'v1.0.0'}`.
**Infer:** this patient's profile (age 63, asymptomatic-adjacent chest pain type,
elevated fasting blood sugar) is a textbook higher-risk presentation, so a
prediction of `disease` with `probability > 0.7` is plausible rather than
surprising — worth sanity-checking a known example like this against domain
intuition before trusting the API on inputs you can't eyeball, exactly the way
Session 4 checked its obesity prediction against the input profile before moving
on.

## Step 6 — Test input validation with a bad request

In [ ]:
bad_patient = dict(sample_patient)
bad_patient["age"] = -5      # invalid: below Field(ge=1)
bad_patient["sex"] = 2       # invalid: outside {0, 1}

resp = client.post("/v1/predict", json=bad_patient)
print(resp.status_code)
import json
print(json.dumps(resp.json(), indent=2))

**Observe:** `422` (not `500`), and a `detail` list with two entries — one for
`age` (`"Input should be greater than or equal to 1"`) and one for `sex`
(`"Input should be less than or equal to 1"`) — each naming the exact field and
constraint that failed.
**Infer:** the request never reached the `predict()` function body at all — Pydantic
rejected it during request parsing, before `model.predict_proba` was ever called.
That's the entire point of Step 3's `Field(ge=..., le=...)` bounds: bad input fails
fast with a specific, machine-readable reason, instead of either crashing inside
the model call (a `500`, much harder for a client to act on) or — worse — silently
returning a confident-looking prediction for physiologically impossible input.

### Realistic failure mode: a schema change that breaks existing clients

Suppose a later retraining adds a fourteenth feature the model now expects. If you
edit `PatientFeatures` in place to require it, every existing client — mobile apps,
other services — that doesn't yet send that field starts getting `422` errors on
every request, with no warning and no migration window.

**Observe:** in the scenario above, error rate on `/v1/predict` would jump from
near-zero to 100% simultaneously across every client, right at deploy time — a
classic sign of a breaking schema change rather than a genuine data or model
problem.
**Infer:** the fix isn't to make the new field optional forever (that just delays
the same problem and lets some requests silently skip a feature the model was
actually trained to expect) — it's to ship the new schema as a new version,
`/v2/predict`, and give existing clients a deprecation window on `/v1/` before
retiring it. That's exactly what Step 7 sets up.

## Step 7 — Version the API alongside the model

`MODEL_VERSION` and the URL prefix (`/v1/`) are two *different* version numbers on
purpose. `MODEL_VERSION` can bump on every retrain (`v1.0.0` → `v1.0.1`) without
any client-visible change, as long as the request/response schema stays the same —
clients don't care that the coefficients changed. The URL prefix only bumps
(`/v1/` → `/v2/`) when the *contract* changes: a new required field, a renamed
response key, a different meaning for an existing field. Conflating the two is a
common mistake — it either forces a breaking URL bump for a routine retrain, or
hides a real breaking change behind a version string nobody checks.

In [ ]:
# Illustrative v2 schema — the fourteenth feature scenario from above.
# Not written to disk here; this cell just shows the *shape* of a non-breaking
# migration: v1 keeps working unchanged, v2 is added alongside it.

v2_sketch = '''
class PatientFeaturesV2(PatientFeatures):
    resting_hr_variability: float = Field(..., ge=0, le=100)

@app.post("/v2/predict", response_model=PredictionResponse)
def predict_v2(features: PatientFeaturesV2):
    ...  # loads a model retrained on the extra feature
'''
print(v2_sketch)
print("/v1/predict keeps serving the original 13-feature model, unchanged.")

**Observe:** that `PatientFeaturesV2` *extends* `PatientFeatures` rather than
replacing it, and that `/v1/predict` isn't touched or removed by this change.
**Infer:** a client on `/v1/` notices nothing the day this ships — it keeps
working exactly as before, on the original model, until you deliberately deprecate
it on a schedule you control (e.g. "`/v1/` sunsets in 90 days, see the response
header `Deprecation: true`"). That's the difference between a versioning scheme
that protects existing clients and one that's just a version number nobody
actually reads.

## Step 8 — Run it for real with `uvicorn`

`TestClient` is enough for local testing, but a real deployment runs the app with
an ASGI server. This is the command a Dockerfile (Session 6) or a CI workflow
(Session 10) would actually invoke.

In [ ]:
# Not run in this notebook -- shown as the real invocation you'd use outside it.
print("uvicorn app:app --host 0.0.0.0 --port 8000")
print()
print("# Then, from another terminal:")
print('curl -X POST http://localhost:8000/v1/predict \\')
print("  -H 'Content-Type: application/json' \\")
print(f"  -d '{sample_patient}'")

**Observe:** the printed `uvicorn` command and the equivalent `curl` call — the
same request body used against `TestClient` in Step 5, just sent over an actual
socket this time.
**Infer:** interactive, browsable docs are also available for free at
`http://localhost:8000/docs` once `uvicorn` is running — FastAPI generates that
page directly from the `PatientFeatures` and `PredictionResponse` Pydantic models
defined in Step 3, so the documentation can never drift out of sync with the actual
validation rules the way a hand-written API doc can.

## What to try next

* Wrap this API in a container the way Session 6 does for its Flask/FastAPI
  services, then deploy it to a Kubernetes cluster so `/health` becomes a real
  liveness probe instead of just a manual check.
* Session 10 builds a GitHub Actions workflow that could run this notebook's
  Step 5/6 tests automatically on every push — a natural next step once this API
  lives in its own repository.
* Compare this hand-built API against Session 8's Flask + SageMaker approach,
  where the model runs on a managed endpoint and the web layer only handles the
  form and the HTTP call — a different split of responsibilities for the same
  underlying problem.
* Add authentication (an API key header, checked with a FastAPI dependency) before
  treating this as production-ready — right now, `/v1/predict` is open to anyone
  who can reach the port.